这份文档总结了你在无固定公网 IP 的环境下，为客户设立安全受限 sFTP 服务的全过程。你可以将其保存，以便日后在其他服务器上快速部署。
🚀 全能 sFTP 设立与内网穿透部署手册
本方案适用于：内网服务器、无公网 IP、需要给外部客户提供安全文件上传通道。
第一阶段：服务器内部 sFTP 环境配置
这一步确保客户只能通过 sFTP 访问，被锁定在指定目录（Chroot），且无法登录服务器命令行。
1. 创建用户组与账号
code
Bash
# 创建 sFTP 专用组
sudo groupadd sftp_users

# 创建用户 (以 client_sftp 为例)，禁止其登录 Shell
sudo useradd -m -g sftp_users -s /usr/sbin/nologin client_sftp

# 设置复杂密码
sudo passwd client_sftp
2. 构建受限目录结构（权限是关键！）
sFTP 的安全机制要求：从根目录到用户主目录的所有父级目录，所有者必须是 root，且权限不能超过 755。
code
Bash
# 1. 创建自定义数据目录
sudo mkdir -p /srv/external_data/client_sftp/files

# 2. 设置 Chroot 根目录权限（必须是 root:root）
sudo chown root:root /srv/external_data/client_sftp
sudo chmod 755 /srv/external_data/client_sftp

# 3. 设置用户真正可以上传文件的子目录（属于用户）
sudo chown client_sftp:sftp_users /srv/external_data/client_sftp/files
sudo chmod 755 /srv/external_data/client_sftp/files

# 4. 修正用户的主目录指向（确保与 Chroot 路径一致）
sudo usermod -d /srv/external_data/client_sftp client_sftp
3. 配置 SSH 服务
编辑配置文件：sudo nano /etc/ssh/sshd_config
在文件末尾添加（或修改）：
code
Ssh
# 针对 sftp_users 组的特定限制
Match Group sftp_users
    ChrootDirectory %h
    ForceCommand internal-sftp
    AllowTcpForwarding yes
    X11Forwarding no
    PasswordAuthentication yes
重启 SSH 服务：
code
Bash
sudo systemctl restart ssh
第二阶段：内网穿透部署（解决无公网 IP）
当服务器处于内网时，通过 Bore 将本地端口映射到公网。
1. 安装 Bore 工具
code
Bash
wget https://github.com/ekzhang/bore/releases/download/v0.5.1/bore-v0.5.1-x86_64-unknown-linux-musl.tar.gz
tar -xf bore-v0.5.1-x86_64-unknown-linux-musl.tar.gz
sudo mv bore /usr/local/bin/
2. 后台运行隧道
使用 nohup 确保退出终端后服务不中断。
code
Bash
# 将本地 22 端口映射到 bore.pub 服务器
sudo nohup bore local 22 --to bore.pub > bore.log 2>&1 &
3. 获取外网连接端口
每次启动时，bore.pub 会分配一个随机端口，通过日志查看：
code
Bash
cat bore.log
找到输出中的： listening at bore.pub:XXXXX（记录下这个 XXXXX 端口号）。
第三阶段：防火墙与安全设置
确保服务器防火墙不会拦截内部转发：
code
Bash
# 允许 SSH 默认端口
sudo ufw allow 22/tcp

# 允许本地回环流量
sudo ufw allow in on lo

# 重载防火墙
sudo ufw reload
第四阶段：交付给客户的信息
将以下格式的信息发送给客户，他们即可通过 WinSCP、FileZilla 等工具连接。
配置项	内容
协议	SFTP (非 FTP)
主机名 (Host)	bore.pub
端口号 (Port)	XXXXX (填 bore.log 中看到的数字)
用户名	client_sftp
密码	你设置的密码
目标文件夹	登录后请进入 /files 文件夹上传
第五阶段：日常维护与排错
1. 监控日志
如果客户反馈连不上，第一时间查看 bore 日志看端口是否发生变化：
code
Bash
tail -f bore.log
2. 检查 SSH 登录记录
查看是否有非法尝试或登录失败的原因：
code
Bash
sudo tail -f /var/log/auth.log
3. 常见故障点
Connection reset: 通常是穿透工具（如 Playit 的 Minecraft 模板）干扰了协议，请确保使用 Bore 或 Ngrok 这种纯 TCP 隧道。
Broken pipe: 100% 是目录权限问题。请检查 /srv/external_data/client_sftp 是否属于 root:root 且权限是否为 755。
Permission denied: 检查客户是否尝试直接在根目录写文件，提醒他们必须进入 files 子目录。
💡 进阶建议：
若要长期给客户提供服务，建议购买一个 $2/年 的廉价域名，并配置 Cloudflare Tunnel (cloudflared)。
优点： 域名固定、端口永远是默认的、不需要手动运行 nohup、支持开机自启，是目前最专业且免费的商业级内网穿透方案。


要找到 2 美元甚至更便宜的域名，通常需要避开流行的 .com 或 .net（这些通常在 10-15 美元），而去选择一些新顶级域名 (New TLDs)。
以下是目前市场上最便宜的域名购买渠道和建议：
1. 推荐的注册商 (Registrars)
这些平台经常有 $0.99 到 $2.00 的首年促销：
Spaceship (强烈推荐):
这是目前业内价格最激进的新注册商。
价格： .top、.xyz、.site、.online 等域名经常在 $1.00 - $2.50 之间。
优点： 界面极其现代，没有乱七八糟的捆绑销售，且自带免费的隐私保护。
Namecheap:
老牌注册商，有一个专门的 99 Cent 促销页。
价格： 经常有 .pw、.icu、.website 等后缀只需 $0.99。
Porkbun:
口碑极好，价格透明。
价格： .xyz 或 .top 经常有首年低价促销。
Dynadot:
经常有针对特定后缀的限时优惠，比如 .link、.click。

athenomics
.cc
+0.00%
$8.26/yr
$3.11

Add to cart